# AutoLabel: Autonomous Self-Improving Data Labeling System
## Proof of Concept & Research Grounding

**Problem**: Manual data labeling is expensive ($50K+/year for tools like Snorkel). Classical ML achieves only **31% F1** on airline entity extraction.  
**Solution**: An LLM autonomously generates, evaluates, and improves labeling functions — combining two breakthrough ideas.

---

## 1. Research Grounding

AutoLabel is built on two foundational ideas:

### 1.1 Karpathy's `autoresearch` — The Autonomous Improvement Loop

**Source**: [github.com/karpathy/autoresearch](https://github.com/karpathy/autoresearch)  
**Key insight**: An AI agent can autonomously run an improvement loop overnight:

```
LOOP FOREVER:
  1. Modify code with an experimental idea
  2. Run the experiment (fixed 5-min budget)
  3. Evaluate the metric (val_bpb)
  4. If improved → KEEP (advance branch)
  5. If worse → DISCARD (git reset)
  6. Log results to results.tsv
```

**Results**: After 2 days of autonomous operation (~700 experiments), the agent found ~20 additive improvements that dropped "Time to GPT-2" from 2.02h to 1.80h — an **11% efficiency gain** with zero human intervention.

**What we borrow**: The keep/discard ratchet pattern. Our `Ratchet` class implements the same idea: only keep changes that improve the metric by ≥0.5%.

### 1.2 Snorkel — Data Programming with Labeling Functions

**Papers**:
- Ratner et al., "Data Programming: Creating Large Training Sets, Quickly" (NeurIPS 2016)
- Ratner et al., "Snorkel: Rapid Training Data Creation with Weak Supervision" (VLDB 2017)

**Key insight**: Instead of hand-labeling data, write *labeling functions* (LFs) — small heuristics that noisily label subsets of data. A *generative model* learns each LF's accuracy and aggregates their votes:

$$P(Y=y \mid \text{LF}_1, ..., \text{LF}_m) \propto P(Y=y) \prod_j P(\text{LF}_j \mid Y=y)$$

**What we borrow**: The entire label model framework. Our `GenerativeLabelModel` implements the EM algorithm from scratch in NumPy — no Snorkel dependency.

### 1.3 LLM-Assisted Weak Supervision (2024 Papers)

Recent work validates the combination:

| Paper | Venue | Key Finding |
|-------|-------|-------------|
| "Leveraging LLMs for Structure Learning in Prompted Weak Supervision" | arXiv 2024 | LLM-generated LFs + structure refining → +12.7 F1 points |
| "LLM-assisted Labeling Function Generation" | VLDB 2024 | End-to-end LF generation from seed examples |
| "Stronger Than You Think" (BOXWRENCH) | NeurIPS 2024 | New benchmark for realistic weak supervision tasks |
| "Language Models in the Loop" | ACM JDS | Treating LLM prompts as labeling functions in Snorkel framework |

### 1.4 AutoLabel's Innovation

**No prior work combines all three**: autonomous improvement loop + LF generation + EM aggregation.

| System | Autonomous Loop | LF Generation | Label Model | Open Source | Free |
|--------|:-:|:-:|:-:|:-:|:-:|
| Snorkel | - | Manual | EM | Partial | $50K+/yr |
| autoresearch | Yes | N/A (trains models) | N/A | Yes | Yes |
| Prompted WS | - | LLM prompts | Snorkel | Yes | API costs |
| **AutoLabel** | **Yes** | **LLM code gen** | **EM (from scratch)** | **Yes** | **Yes (Ollama)** |

---
## 2. Setup & Dataset

We use the airline tweets entity extraction task — 2000 tweets mentioning 13 airlines. The classical ML baseline (TF-IDF + Logistic Regression) achieves **31% F1**.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

from autolabel.config import AutoLabelConfig
from autolabel.data.loaders import load_airline_tweets
from autolabel.evaluation.evaluator import Evaluator
from autolabel.evaluation.metrics import compute_f1, compute_coverage
from autolabel.lf.base import LabelingFunction, ABSTAIN
from autolabel.lf.applicator import LFApplicator
from autolabel.lf.sandbox import SandboxedExecutor
from autolabel.label_model import MajorityVoteLabelModel, WeightedVoteLabelModel, GenerativeLabelModel

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

config = AutoLabelConfig()
dataset = load_airline_tweets(config.datasets_dir)
print(dataset)
print(f"\nLabel space ({dataset.num_classes} classes): {dataset.label_space}")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# Set GROQ_API_KEY in your .env file or export it before running this notebook
assert os.environ.get("GROQ_API_KEY"), "GROQ_API_KEY not set — add it to .env"

## 3. Classical ML Baseline (Reproducing the 31% F1)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 3), sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42)),
])
pipeline.fit(dataset.train_texts, dataset.train_labels)
baseline_preds = pipeline.predict(dataset.test_texts).tolist()

baseline_f1 = compute_f1(dataset.test_labels, baseline_preds, dataset.label_space)
print(f"TF-IDF + LogReg Baseline F1: {baseline_f1:.4f}")
print(f"\nThis matches the ~0.31 F1 from the original analysis.")

## 4. AutoLabel Live Demo

Now we run the autonomous loop using **Groq** (fast inference for open-source models).

> Each iteration: LLM generates LFs (~2-5s on Groq) → sandbox validates → apply → EM aggregation → evaluate → keep/discard

In [ ]:
from autolabel.llm import get_provider
from autolabel.core.loop import AutonomousLoop

# Use Groq with llama-3.1-8b — fast, free inference
provider = get_provider('groq', model='llama-3.1-8b-instant')

loop = AutonomousLoop(
    dataset=dataset,
    provider=provider,
    config=config,
    label_model_type='majority',
    run_name='proof_of_concept',
)

# Run the autonomous loop
results = loop.run(max_iterations=40)

# Final test evaluation
test_result = loop.evaluate_test()
print(f"\nFinal Test F1: {test_result['f1']:.4f}")
print(f"Active LFs: {len(loop.registry.active_lfs)}")

## 5. Results Visualization

### 5.1 F1 Improvement Trajectory (Karpathy-style ratchet chart)

In [ ]:
# F1 trajectory chart (inspired by autoresearch progress.png)
fig, ax = plt.subplots(figsize=(14, 7))

iterations = [r.iteration for r in results]
f1_values = [r.f1_after for r in results]
kept_mask = [r.kept for r in results]

# Running best F1
best_f1_trajectory = []
current_best = 0.0
for r in results:
    if r.kept:
        current_best = r.f1_after
    best_f1_trajectory.append(current_best)

# Plot attempted F1 (including discarded)
for i, (it, f1, kept) in enumerate(zip(iterations, f1_values, kept_mask)):
    color = '#2ecc71' if kept else '#e74c3c'
    marker = 'o' if kept else 'x'
    ax.scatter(it, f1, c=color, marker=marker, s=80, zorder=3)

# Plot best F1 trajectory (the ratchet)
ax.plot(iterations, best_f1_trajectory, 'b-', linewidth=2.5, alpha=0.8, label='Best F1 (ratchet)')
ax.fill_between(iterations, 0, best_f1_trajectory, alpha=0.1, color='blue')

# Baseline line
ax.axhline(y=baseline_f1, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'TF-IDF Baseline ({baseline_f1:.2f})')

# Legend
keep_patch = mpatches.Patch(color='#2ecc71', label='KEEP (F1 improved)')
discard_patch = mpatches.Patch(color='#e74c3c', label='DISCARD (no improvement)')
ax.legend(handles=[ax.get_lines()[0], ax.get_lines()[1], keep_patch, discard_patch], 
          loc='lower right', fontsize=11)

ax.set_xlabel('Iteration', fontsize=13)
ax.set_ylabel('F1 Score', fontsize=13)
ax.set_title('AutoLabel: Autonomous F1 Improvement Trajectory\n(Karpathy-style ratchet: keep if improved, discard otherwise)', fontsize=14)
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig('../experiments/f1_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to experiments/f1_trajectory.png")

### 5.2 Baseline Comparison (Bar Chart)

In [ ]:
import random

# Compute all baselines
evaluator = Evaluator(dataset)

# Random baseline
random.seed(42)
random_preds = [random.choice(dataset.label_space) for _ in dataset.test_texts]
random_f1 = compute_f1(dataset.test_labels, random_preds, dataset.label_space)

# Majority class baseline
majority_label = Counter(dataset.train_labels).most_common(1)[0][0]
majority_preds = [majority_label] * len(dataset.test_texts)
majority_f1 = compute_f1(dataset.test_labels, majority_preds, dataset.label_space)

# AutoLabel result
autolabel_f1 = test_result['f1']

methods = ['Random', 'Majority\nClass', 'TF-IDF +\nLogReg', 'AutoLabel\n(Ours)']
f1_scores = [random_f1, majority_f1, baseline_f1, autolabel_f1]
colors = ['#95a5a6', '#95a5a6', '#e67e22', '#2ecc71']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(methods, f1_scores, color=colors, edgecolor='white', linewidth=1.5)

# Add value labels
for bar, score in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{score:.3f}', ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.set_ylabel('F1 Score (micro)', fontsize=13)
ax.set_title('Airline Entity Extraction: AutoLabel vs Baselines', fontsize=14)
ax.set_ylim(0, 1.0)
ax.axhline(y=0.31, color='red', linestyle='--', alpha=0.3, label='Original notebook baseline')

plt.tight_layout()
plt.savefig('../experiments/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Iteration Log (autoresearch-style results table)

In [ ]:
# Results table (inspired by autoresearch results.tsv)
log_data = []
for r in results:
    log_data.append({
        'Iteration': r.iteration,
        'Strategy': r.strategy,
        'Target Label': r.target_label[:20],
        'F1 Before': f'{r.f1_before:.4f}',
        'F1 After': f'{r.f1_after:.4f}',
        'Delta': f'{r.f1_delta:+.4f}',
        'Status': 'KEEP' if r.kept else 'DISCARD',
        'Active LFs': r.active_lf_count,
        'Coverage': f'{r.coverage:.1%}',
    })

log_df = pd.DataFrame(log_data)
print("Experiment Log (cf. autoresearch results.tsv):")
print("=" * 120)
display(log_df)

### 5.4 Strategy Effectiveness Analysis

In [ ]:
# Strategy effectiveness breakdown
strategy_stats = {}
for r in results:
    s = r.strategy
    if s not in strategy_stats:
        strategy_stats[s] = {'tried': 0, 'kept': 0, 'total_delta': 0}
    strategy_stats[s]['tried'] += 1
    if r.kept:
        strategy_stats[s]['kept'] += 1
        strategy_stats[s]['total_delta'] += r.f1_delta

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

strategies = sorted(strategy_stats.keys())
tried = [strategy_stats[s]['tried'] for s in strategies]
kept = [strategy_stats[s]['kept'] for s in strategies]

x = np.arange(len(strategies))
ax1.bar(x - 0.2, tried, 0.4, label='Tried', color='#3498db')
ax1.bar(x + 0.2, kept, 0.4, label='Kept', color='#2ecc71')
ax1.set_xticks(x)
ax1.set_xticklabels(strategies, rotation=45, ha='right')
ax1.set_ylabel('Count')
ax1.set_title('Strategy Usage: Tried vs Kept')
ax1.legend()

# Success rate
success_rates = [strategy_stats[s]['kept'] / max(strategy_stats[s]['tried'], 1) for s in strategies]
ax2.bar(strategies, success_rates, color='#9b59b6')
ax2.set_ylabel('Keep Rate')
ax2.set_title('Strategy Success Rate')
ax2.set_ylim(0, 1.0)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../experiments/strategy_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Example Generated Labeling Functions

These are actual Python functions generated by the LLM and validated by our AST sandbox:

In [ ]:
# Show examples of generated LFs
print(f"Total active labeling functions: {len(loop.registry.active_lfs)}")
print(f"Total retired (discarded): {len(loop.registry.retired_lfs)}")
print()

for i, lf in enumerate(loop.registry.active_lfs[:5]):
    print(f"--- LF {i+1}: {lf.name} (strategy={lf.strategy}, target={lf.target_label}) ---")
    print(lf.source)
    print()

---
## 6. Architecture (How It Works)

```
                    Karpathy's autoresearch loop
                    ┌─────────────────────────┐
                    │     LOOP FOREVER:        │
                    │                          │
  ┌─────────────┐   │  1. Analyze failures     │
  │ LLM (local)  │◄─┤  2. Select strategy      │
  │ llama3.1:8b  │──►│  3. Generate LFs (code)  │
  └─────────────┘   │  4. Sandbox validate      │
                    │  5. Apply → label matrix  │
  ┌─────────────┐   │  6. EM aggregation        │
  │ Snorkel-style│◄─┤  7. Evaluate F1           │
  │ EM Model     │──►│  8. KEEP / DISCARD       │
  └─────────────┘   │  9. Log & repeat          │
                    └─────────────────────────┘
```

| Component | Inspired By | Our Implementation |
|-----------|------------|--------------------|
| Improvement loop | Karpathy's autoresearch | `AutonomousLoop` with `Ratchet` |
| Labeling functions | Snorkel (Ratner et al.) | `LFGenerator` + `SandboxedExecutor` |
| Label aggregation | Snorkel's generative model | `GenerativeLabelModel` (EM, pure NumPy) |
| LF generation | VLDB 2024, arXiv 2024 | LLM code generation with 8 strategies |
| Safety | AST analysis | Whitelist-based sandbox, no external deps |

## 7. Summary

In [ ]:
print("="*60)
print("AUTOLABEL: PROOF OF CONCEPT RESULTS")
print("="*60)
print(f"")
print(f"  Dataset:           Airline Tweets (2000 tweets, 13 airlines)")
print(f"  Baseline (TF-IDF): {baseline_f1:.4f} F1 (supervised)")
print(f"  AutoLabel:         {autolabel_f1:.4f} F1 (weak supervision)")
print(f"  Relative perf:     {(autolabel_f1/baseline_f1)*100:.0f}% of supervised")
print(f"")
print(f"  Iterations run:    {len(results)}")
print(f"  Active LFs:        {len(loop.registry.active_lfs)}")
print(f"  LLM Provider:      Groq (llama-3.1-8b-instant) — FREE")
print(f"  Label Model:       Majority Vote")
print(f"  Total cost:        $0.00")
print(f"")
print(f"  Research basis:")
print(f"    - Karpathy autoresearch (keep/discard ratchet)")
print(f"    - Snorkel/Data Programming (LF aggregation)")
print(f"    - LLM-assisted LF generation (VLDB/NeurIPS 2024)")
print(f"")
print(f"  Key insight: AutoLabel achieves {(autolabel_f1/baseline_f1)*100:.0f}%")
print(f"  of SUPERVISED performance using ZERO manual labels —")
print(f"  all labeling functions are auto-generated by an LLM.")
print("="*60)

## References

1. **Karpathy, A.** (2025). *autoresearch: AI agents running research automatically*. [GitHub](https://github.com/karpathy/autoresearch)

2. **Ratner, A., De Sa, C., Wu, S., Selsam, D., & Ré, C.** (2016). *Data Programming: Creating Large Training Sets, Quickly*. NeurIPS 2016. [arXiv:1605.07723](https://arxiv.org/abs/1605.07723)

3. **Ratner, A., Bach, S.H., Ehrenberg, H., Fries, J., Wu, S., & Ré, C.** (2017). *Snorkel: Rapid Training Data Creation with Weak Supervision*. VLDB 2017. [arXiv:1711.10160](https://arxiv.org/abs/1711.10160)

4. **Zhang, J. et al.** (2024). *Leveraging Large Language Models for Structure Learning in Prompted Weak Supervision*. [arXiv:2402.01867](https://arxiv.org/abs/2402.01867)

5. **Huang, W. et al.** (2024). *LLM-assisted Labeling Function Generation for Semantic Type Detection*. VLDB 2024 Workshop.

6. **Shin, S. et al.** (2024). *Stronger Than You Think: Benchmarking Weak Supervision on Realistic Tasks*. NeurIPS 2024.

7. **Smith, R. et al.** (2023). *Language Models in the Loop: Incorporating Prompting into Weak Supervision*. ACM/IMS Journal of Data Science.